**Система:** телефонная книга  
**База данных:** SQLite  
**ORM:** Flask-SQLAlchemy  
**Поля записи:** номер телефона, сотовый оператор, имя владельца, остаток средств  
**Возможности:** просмотр, добавление, редактирование, удаление, поиск, сортировка, фильтрация и REST API

На главной странице рассчитывается минимальный остаток средств среди всех записей.

## 0. Установка и импорт библиотек

In [ ]:
!pip -q install flask flask-sqlalchemy

In [ ]:
from flask import Flask, request, jsonify, render_template_string
from flask_sqlalchemy import SQLAlchemy
from sqlalchemy import func
from pathlib import Path

## 1. Конфигурация приложения и ORM-модель

In [ ]:
app = Flask(__name__)

DB_PATH = Path("phones.db").resolve()

app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{DB_PATH}"
app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False
app.config["SECRET_KEY"] = "phone-project-key"

db = SQLAlchemy(app)


class Phone(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    phone_number = db.Column(
        db.String(20),
        unique=True,
        nullable=False,
        index=True
    )
    operator = db.Column(
        db.String(50),
        nullable=False,
        index=True
    )
    owner_name = db.Column(
        db.String(100),
        nullable=False,
        index=True
    )
    balance = db.Column(
        db.Float,
        nullable=False,
        index=True
    )

    def to_dict(self):
        return {
            "id": self.id,
            "phone_number": self.phone_number,
            "operator": self.operator,
            "owner_name": self.owner_name,
            "balance": self.balance,
        }

    def __repr__(self):
        return (
            f"Phone(id={self.id}, "
            f"phone_number='{self.phone_number}', "
            f"operator='{self.operator}', "
            f"owner_name='{self.owner_name}', "
            f"balance={self.balance})"
        )

## 2. Создание и заполнение базы данных

In [ ]:
sample_data = [
    {
        "phone_number": "+79161234567",
        "operator": "МТС",
        "owner_name": "Иван Петров",
        "balance": 150.50,
    },
    {
        "phone_number": "+79031112233",
        "operator": "Билайн",
        "owner_name": "Мария Сидорова",
        "balance": 89.30,
    },
    {
        "phone_number": "+79254445566",
        "operator": "МегаФон",
        "owner_name": "Анна Козлова",
        "balance": 245.75,
    },
    {
        "phone_number": "+79058889900",
        "operator": "Теле2",
        "owner_name": "Петр Иванов",
        "balance": 56.80,
    },
    {
        "phone_number": "+79167778899",
        "operator": "Yota",
        "owner_name": "Ольга Новикова",
        "balance": 320.15,
    },
    {
        "phone_number": "+79019998877",
        "operator": "МТС",
        "owner_name": "Александр Смирнов",
        "balance": 180.25,
    },
    {
        "phone_number": "+79253334455",
        "operator": "Билайн",
        "owner_name": "Екатерина Волкова",
        "balance": 95.60,
    },
    {
        "phone_number": "+79168889900",
        "operator": "МегаФон",
        "owner_name": "Дмитрий Орлов",
        "balance": 310.40,
    },
]


def init_db():
    with app.app_context():
        db.drop_all()
        db.create_all()

        for item in sample_data:
            db.session.add(Phone(**item))

        db.session.commit()


init_db()

with app.app_context():
    print("Количество записей:", Phone.query.count())
    print(
        "Минимальный остаток:",
        db.session.query(func.min(Phone.balance)).scalar()
    )

## 3. HTML-интерфейс

In [ ]:
PAGE_TEMPLATE = """
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Телефонная книга</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 1100px;
            margin: 30px auto;
            padding: 0 20px;
        }

        table {
            width: 100%;
            border-collapse: collapse;
            margin-top: 20px;
        }

        th, td {
            border: 1px solid #bbb;
            padding: 9px;
            text-align: left;
        }

        th {
            background: #eee;
        }

        .metric {
            padding: 15px;
            border: 1px solid #999;
            margin: 15px 0;
        }
    </style>
</head>
<body>
    <h1>Телефонная книга</h1>

    {% if min_balance is not none %}
        <div class="metric">
            Минимальный остаток средств:
            <strong>{{ "%.2f"|format(min_balance) }} ₽</strong>
        </div>
    {% endif %}

    {% if phones %}
        <table>
            <thead>
                <tr>
                    <th>ID</th>
                    <th>Номер телефона</th>
                    <th>Оператор</th>
                    <th>Владелец</th>
                    <th>Остаток, ₽</th>
                </tr>
            </thead>
            <tbody>
                {% for phone in phones %}
                    <tr>
                        <td>{{ phone.id }}</td>
                        <td>{{ phone.phone_number }}</td>
                        <td>{{ phone.operator }}</td>
                        <td>{{ phone.owner_name }}</td>
                        <td>{{ "%.2f"|format(phone.balance) }}</td>
                    </tr>
                {% endfor %}
            </tbody>
        </table>
    {% endif %}
</body>
</html>
"""

In [ ]:
@app.route("/")
def index():
    min_balance = db.session.query(
        func.min(Phone.balance)
    ).scalar()

    phones_count = Phone.query.count()

    return jsonify({
        "phones_count": phones_count,
        "min_balance": min_balance if min_balance is not None else 0,
    })


@app.route("/phones")
def view_phones():
    phones = Phone.query.all()

    min_balance = db.session.query(
        func.min(Phone.balance)
    ).scalar()

    return render_template_string(
        PAGE_TEMPLATE,
        phones=phones,
        min_balance=min_balance,
    )

## 4. CRUD-операции

In [ ]:
@app.route("/api/phones", methods=["GET", "POST"])
def api_phones():
    if request.method == "GET":
        phones = Phone.query.all()
        return jsonify([phone.to_dict() for phone in phones])

    data = request.get_json(force=True)

    required = [
        "phone_number",
        "operator",
        "owner_name",
        "balance",
    ]

    for field in required:
        if field not in data:
            return jsonify({"error": f"Поле {field} отсутствует"}), 400

    phone = Phone(
        phone_number=str(data["phone_number"]).strip(),
        operator=str(data["operator"]).strip(),
        owner_name=str(data["owner_name"]).strip(),
        balance=float(data["balance"]),
    )

    db.session.add(phone)

    try:
        db.session.commit()
    except Exception as exc:
        db.session.rollback()
        return jsonify({"error": str(exc)}), 400

    return jsonify(phone.to_dict()), 201


@app.route("/api/phones/<int:phone_id>", methods=["PUT", "DELETE"])
def api_phone_by_id(phone_id):
    phone = db.session.get(Phone, phone_id)

    if phone is None:
        return jsonify({"error": "Запись не найдена"}), 404

    if request.method == "DELETE":
        db.session.delete(phone)
        db.session.commit()
        return jsonify({"status": "deleted", "id": phone_id})

    data = request.get_json(force=True)

    phone.phone_number = data.get(
        "phone_number",
        phone.phone_number
    )
    phone.operator = data.get(
        "operator",
        phone.operator
    )
    phone.owner_name = data.get(
        "owner_name",
        phone.owner_name
    )

    if "balance" in data:
        phone.balance = float(data["balance"])

    db.session.commit()

    return jsonify(phone.to_dict())

## 5. Поиск, сортировка и фильтрация

In [ ]:
@app.route("/api/search/<path:phone_number>")
def api_search(phone_number):
    phone = Phone.query.filter_by(
        phone_number=phone_number
    ).first()

    if phone is None:
        return jsonify({"error": "Запись не найдена"}), 404

    return jsonify(phone.to_dict())


@app.route("/api/sorted/operator")
def api_sorted_operator():
    phones = Phone.query.order_by(
        Phone.operator.asc()
    ).all()

    return jsonify([
        phone.to_dict()
        for phone in phones
    ])


@app.route("/api/filter/owner/<string:prefix>")
def api_filter_owner(prefix):
    phones = Phone.query.filter(
        Phone.owner_name.startswith(prefix)
    ).all()

    return jsonify([
        phone.to_dict()
        for phone in phones
    ])


@app.route("/api/min_balance/<string:owner_name>")
def api_min_balance(owner_name):
    value = db.session.query(
        func.min(Phone.balance)
    ).filter(
        Phone.owner_name < owner_name
    ).scalar()

    return jsonify({
        "owner_name_limit": owner_name,
        "min_balance": value if value is not None else 0,
    })

## 6. Проверка веб-сервиса

In [ ]:
client = app.test_client()

print("Главная:")
print(client.get("/").get_json())

print("\nПоиск:")
print(
    client.get(
        "/api/search/+79161234567"
    ).get_json()
)

print("\nСортировка по оператору:")
sorted_data = client.get(
    "/api/sorted/operator"
).get_json()

for item in sorted_data:
    print(item["operator"], item["phone_number"])

print("\nФильтрация владельцев на 'А':")
print(
    client.get(
        "/api/filter/owner/А"
    ).get_json()
)

print("\nМинимальный остаток при ограничении имени:")
print(
    client.get(
        "/api/min_balance/М"
    ).get_json()
)

## 7. Добавление, редактирование и удаление через API

In [ ]:
new_record = {
    "phone_number": "+79990001122",
    "operator": "МТС",
    "owner_name": "Сергей Николаев",
    "balance": 210.40,
}

response = client.post(
    "/api/phones",
    json=new_record
)

created = response.get_json()
print("Добавление:", created)

created_id = created["id"]

response = client.put(
    f"/api/phones/{created_id}",
    json={"balance": 333.30}
)

print("Редактирование:", response.get_json())

response = client.delete(
    f"/api/phones/{created_id}"
)

print("Удаление:", response.get_json())

## Итог

In [ ]:
with app.app_context():
    min_balance = db.session.query(
        func.min(Phone.balance)
    ).scalar()

    print("Количество записей:", Phone.query.count())
    print(f"Минимальный остаток: {min_balance:.2f} ₽")
    print("База данных:", DB_PATH)